# ✈️ Extracción y Procesamiento Inicial de Datos de Tráfico Aéreo (OpenSky)

La disponibilidad de datos aeronáuticos en tiempo real permite analizar patrones de movilidad aérea, comportamiento de aeronaves y variaciones operativas a escala global.  
Este proyecto desarrolla un **pipeline ETL** que ingesta información del endpoint público de **OpenSky Network**, centrado en capturar el estado actual de miles de vuelos activos en simultáneo.

La API proporciona variables clave como:

- identificador **ICAO24**  
- país de origen  
- latitud / longitud  
- altitudes barométrica y geométrica  
- velocidad, rumbo, tasa vertical  
- estado en tierra o en vuelo  
- timestamp del servidor  

Estos datos se transforman y almacenan en un **Data Lake local** siguiendo la arquitectura **Bronze → Silver → Gold**, lo que permite realizar análisis históricos, construir métricas aeronáuticas y preparar la futura migración a entornos de nube (Azure).

## Objetivos

**Extracción (Bronze):**
- Consumir el endpoint `states/all` de OpenSky.  
- Normalizar la estructura JSON y convertirla en tabla.  
- Incorporar timestamps (servidor y extracción).  
- Guardar los datos crudos en **Delta Lake**.

**Transformación (Silver):**
- Limpiar valores faltantes y tipos de datos.  
- Estandarizar columnas y coordenadas.  
- Preparar la tabla para análisis temporal.

**Métricas (Gold):**
- Crear features de movilidad aérea: altitud efectiva, variación de velocidad, indicadores de vuelo/estacionamiento, etc.  
- Generar datasets optimizados para visualización y análisis exploratorio.

## Alcance y supuestos

- Se utilizan exclusivamente datos públicos provistos por OpenSky Network.  
- La extracción se realiza bajo límites de la API pública (sin autenticación obligatoria).  
- El objetivo es **práctico y educativo**, orientado al portfolio de Ingeniería de Datos.

## Reproducibilidad

- Dependencias detalladas en `requirements.txt`.  
- Las rutas del Data Lake se configuran en `pipeline.conf`.  
- Todas las funciones auxiliares se encuentran en `src/etl_utils.py`.

---

**Estructura del notebook:**

0) Configuración inicial  
1) Extracción del endpoint `states/all`  
2) Normalización del JSON  
3) Limpieza y estandarización mínima  
4) Almacenamiento en Delta Lake (capa Bronze)  
5) Verificación y vista preliminar de los datos  

## 0. Configuración inicial

En este paso se importan todas las librerías necesarias y las funciones auxiliares definidas en `etl_utils.py`.  
Este enfoque permite mantener el notebook **ordenado, modular y fácilmente reproducible**, centralizando en un único módulo las operaciones comunes del pipeline ETL: extracción desde la API pública de **OpenSky Network**, normalización del JSON, estandarización de columnas y escritura en las distintas capas del **Data Lake local** (Bronze → Silver → Gold).

El objetivo de esta sección es garantizar que todas las dependencias estén correctamente cargadas antes de iniciar el proceso de extracción y almacenamiento.

In [1]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

# Importar funciones auxiliares
from etl_utils import *

# Librerías comunes
import pandas as pd

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


## 1. Autenticación y lectura de configuración

La configuración del proyecto se administra mediante el archivo `pipeline.conf`,  
que centraliza parámetros como:
- la **URL base** de la API de OpenSky Network  
- credenciales opcionales para *Basic Auth* (en caso de usarse)  
- rutas del **Data Lake local**

Aunque la API pública de OpenSky no requiere autenticación obligatoria,almacenar parámetros en un archivo de configuración permite:
- mantener el notebook limpio  
- evitar credenciales expuestas en el código  
- facilitar la migración futura a servicios en la nube (Azure Key Vault)

El archivo se lee mediante `ConfigParser`, lo que permite obtener los valores  
en forma segura y reusable.


In [3]:
# Se instancia el parser y se lee el archivo de configuración
from configparser import ConfigParser

parser = ConfigParser()
parser.read("../pipeline.conf")

['../pipeline.conf']

In [4]:
# Parámetros de conexión
api_config = parser["api-opensky"]
base_url = api_config["base_url"]

In [5]:
print("📄 Configuración cargada correctamente.")
print(f"URL base: {base_url}")

📄 Configuración cargada correctamente.
URL base: https://opensky-network.org/api/states/all


In [6]:
# Prueba de conexión a la API OpenSky
response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    print(f"La petición fue exitosa. Tipo de respuesta: {type(data)}")

    # Claves principales del JSON
    print(f"Claves principales recibidas: {list(data.keys())[:5]}")

    # Inspección parcial de 'states'
    print("\nPrimeras 2 aeronaves registradas:")
    pprint(data["states"][:2])

else:
    print(f"❌ Error en la petición: {response.status_code}, {response.content}")
print("✅ Prueba de conexión a la API realizada.")

La petición fue exitosa. Tipo de respuesta: <class 'dict'>
Claves principales recibidas: ['time', 'states']

Primeras 2 aeronaves registradas:
[['a8aac8',
  'DAL391  ',
  'United States',
  1763116186,
  1763116188,
  -85.422,
  34.1467,
  6393.18,
  False,
  254.94,
  107.74,
  -18.21,
  None,
  6614.16,
  '3257',
  False,
  0],
 ['80162c',
  'AXB1214 ',
  'India',
  1763116193,
  1763116193,
  73.2766,
  18.9267,
  3048,
  False,
  129.64,
  90.23,
  -4.55,
  None,
  3139.44,
  None,
  False,
  0]]
✅ Prueba de conexión a la API realizada.


## 2. Capa Bronze — Extracción y almacenamiento de datos crudos

En esta etapa se realiza la **extracción directa de datos** desde la API pública de **OpenSky Network**, que provee información en tiempo real sobre aeronaves detectadas a nivel global.  
El objetivo es obtener el conjunto completo de registros tal como es devuelto por el endpoint `states/all` y conservarlo en su forma más fiel dentro de la capa **🟤 Bronze** del Data Lake.

Los datos obtenidos incluyen:

- Identificador único **ICAO24**  
- Indicativo de llamada (**callsign**)  
- País de origen  
- Posición geográfica (latitud, longitud)  
- Altitudes barométrica y geométrica  
- Velocidad, rumbo y tasa vertical  
- Estado operativo (en tierra o en vuelo)  
- Timestamp del servidor (`time`) correspondiente a la captura

Cada aeronave es representada inicialmente como una lista ordenada de 17 elementos, por lo que en esta etapa se prioriza **preservar los datos crudos** antes de aplicar procesos de estructuración o limpieza.

Los datos se almacenan en **formato Delta Lake**, dentro del directorio:

`data/etl_datalake/bronze/api_opensky/`

empleando el modo **`overwrite`**, ya que la información corresponde a un snapshot puntual del estado global del tráfico aéreo y puede reemplazarse completamente en cada actualización.


In [7]:
# Extracción desde la API usando la función auxiliar
json_data = get_opensky_states()

# Vista preliminar de claves del JSON
print("Claves principales del JSON:", list(json_data.keys()))
print("Cantidad de aeronaves:", len(json_data["states"]))

Claves principales del JSON: ['time', 'states']
Cantidad de aeronaves: 5719


### 2.1 Extracción de datos estáticos

Se realiza una *ingesta full* para recursos estáticos de la API de OpenSky Network, que ofrecen información descriptiva sobre aeronaves y sus características (por ejemplo, tipo de aeronave, modelo o categoría operativa).

Dado que estos datos cambian de forma esporádica, se los extrae por completo en cada ejecución y se sobrescribe la versión previa (mode="overwrite").  
Este enfoque simplifica el proceso, evita duplicados y garantiza que siempre se disponga de la versión más reciente sin necesidad de rastrear cambios incrementales.

Los datos obtenidos se almacenan en la capa 🟤 *Bronze* en formato *Delta Lake*, sobrescribiendo en cada ejecución para mantener consistencia y simplicidad en el pipeline.

### 2.2 Extracción de datos dinámicos

Para los datos dinámicos se utiliza la información proveniente del endpoint `states/all`, que ofrece el estado en tiempo real de miles de aeronaves a nivel global.  
Dado que estos datos se actualizan constantemente, se captura un *snapshot* del tráfico aéreo en cada ejecución.

En este caso no se aplica una actualización incremental, ya que el conjunto de aeronaves presentes varía continuamente y no existe un identificador temporal que permita rastrear cambios de manera estricta.  
Por ello se guarda cada ejecución como un registro independiente, preservando la trazabilidad de cada captura.

Los snapshots se almacenan en la capa 🟤 *Bronze* en formato *Delta Lake*, permitiendo conservar la historia de ingestas y habilitando futuras consultas temporales o análisis evolutivos del tráfico aéreo.